## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# To ensure consistent langdetect results
DetectorFactory.seed = 0


## 2. Load Data

In [ ]:
# Load the data
reddit_df = pd.read_csv('data/raw/reddit_political.csv')
yt_df1 = pd.read_csv('data/raw/youtube_actual_politic.csv')
yt_df2 = pd.read_csv('data/raw/youtube_bbc_gaza.csv')

youtube_df = pd.concat([yt_df1, yt_df2], ignore_index=True)

print("Reddit shape:", reddit_df.shape)
print("YouTube shape:", youtube_df.shape)


## 3. Data Cleaning (Deduplication & Null Handling)

In [ ]:
# Null handling
reddit_df.dropna(subset=['title'], inplace=True)
reddit_df['selftext'].fillna('', inplace=True)
youtube_df.dropna(subset=['text'], inplace=True)

# Combine titles and text for Reddit
reddit_df['full_text'] = reddit_df['title'] + ' ' + reddit_df['selftext']

# Deduplication
reddit_df.drop_duplicates(subset=['full_text'], inplace=True)
youtube_df.drop_duplicates(subset=['text'], inplace=True)

print("Cleaned Reddit shape:", reddit_df.shape)
print("Cleaned YouTube shape:", youtube_df.shape)


## 4. Language Detection

In [ ]:
# Language detection
def safe_detect(text):
    try:
        if len(str(text).strip()) > 2:
            return detect(str(text))
    except LangDetectException:
        pass
    return 'unknown'

print("Detecting languages for Reddit...")
# Taking a sample for performance, or full dataframe if possible
reddit_df['lang'] = reddit_df['full_text'].apply(safe_detect)
print("Detecting languages for YouTube...")
youtube_df['lang'] = youtube_df['text'].apply(safe_detect)

# Filter English
reddit_df = reddit_df[reddit_df['lang'] == 'en'].copy()
youtube_df = youtube_df[youtube_df['lang'] == 'en'].copy()

print("English Reddit shape:", reddit_df.shape)
print("English YouTube shape:", youtube_df.shape)


## 5. We/Them Baseline (Lexicon Match)

In [ ]:
# We/Them Baseline
we_words = ['we', 'us', 'our', 'ours', 'ourselves']
them_words = ['they', 'them', 'their', 'theirs', 'themselves']

import re

def count_words(text, word_list):
    text = str(text).lower()
    return sum(len(re.findall(r'\b' + word + r'\b', text)) for word in word_list)

reddit_df['we_count'] = reddit_df['full_text'].apply(lambda x: count_words(x, we_words))
reddit_df['them_count'] = reddit_df['full_text'].apply(lambda x: count_words(x, them_words))

youtube_df['we_count'] = youtube_df['text'].apply(lambda x: count_words(x, we_words))
youtube_df['them_count'] = youtube_df['text'].apply(lambda x: count_words(x, them_words))

# Basic Negativity Lexicon (for baseline before Perspective API in Phase 3)
negative_words = ['bad', 'worst', 'hate', 'stupid', 'idiot', 'fake', 'liar', 'worse', 'terrible', 'awful', 'angry']
reddit_df['negativity_score'] = reddit_df['full_text'].apply(lambda x: count_words(x, negative_words))
youtube_df['negativity_score'] = youtube_df['text'].apply(lambda x: count_words(x, negative_words))

reddit_df[['full_text', 'we_count', 'them_count', 'negativity_score']].head()


## 6. Correlation: Negativity x Engagement

In [ ]:
# Spearman Correlation: Negativity x Engagement
reddit_engagement = reddit_df['score'] + reddit_df['num_comments']
rho_reddit, p_reddit = spearmanr(reddit_df['negativity_score'], reddit_engagement)

youtube_engagement = youtube_df['like_count'] # assuming comments don't have sub-comments explicitly loaded, else use replies
rho_yt, p_yt = spearmanr(youtube_df['negativity_score'], youtube_engagement)

print(f"Reddit Negativity x Engagement Spearman: rho={rho_reddit:.4f}, p={p_reddit:.4f}")
print(f"YouTube Negativity x Engagement Spearman: rho={rho_yt:.4f}, p={p_yt:.4f}")


## 7. Plots: Time-of-day & Engagement Distribution

In [ ]:
# Time of Day Distribution
reddit_df['created_utc'] = pd.to_datetime(reddit_df['created_utc'])
reddit_df['hour'] = reddit_df['created_utc'].dt.hour

plt.figure(figsize=(10, 5))
sns.histplot(reddit_df['hour'], bins=24, kde=True)
plt.title('Reddit: Time-of-day Distribution (Posting Hour)')
plt.xlabel('Hour of Day (UTC)')
plt.ylabel('Count')
plt.show()

# You can add reply-depth analysis if the data contains parent_id or nested structures.
